# Redes Neuronales Convolucionales (CNN)

## ¿Cuál es el objetivo de una CNN?

Las **Redes Neuronales Convolucionales** (Convolutional Neural Networks, CNN) son un tipo de red neuronal diseñada específicamente para procesar datos con **estructura espacial**, como imágenes.

El objetivo de una CNN es **aprender automáticamente características jerárquicas** de los datos:

- **Capas iniciales**: detectan características simples (bordes, esquinas, texturas)
- **Capas intermedias**: combinan características simples en patrones más complejos (ojos, narices, ruedas)
- **Capas finales**: reconocen objetos completos (caras, autos, animales)

**¿Por qué no usar una red fully-connected (MLP) para imágenes?**

Una imagen de solo $28 \times 28$ píxeles en escala de grises tiene $784$ valores. Si la primera capa oculta tiene $128$ neuronas, necesitamos $784 \times 128 = 100{,}352$ pesos solo en la primera capa. Para una imagen de $224 \times 224 \times 3$ (color), serían $224 \times 224 \times 3 \times 128 = 19{,}267{,}584$ pesos.

Las CNNs resuelven este problema con tres ideas clave:

| Concepto | Descripción | Beneficio |
|---|---|---|
| **Conexiones locales** | Cada neurona solo ve una región pequeña de la entrada | Reduce drásticamente los parámetros |
| **Pesos compartidos** | El mismo filtro se aplica en toda la imagen | Los patrones se detectan sin importar su posición |
| **Invariancia a la traslación** | Un gato es un gato sin importar dónde esté en la imagen | Generalización robusta |

Una CNN típica alterna tres tipos de operaciones:

$$
\text{Imagen} \xrightarrow{\text{Conv}} \xrightarrow{\text{ReLU}} \xrightarrow{\text{Pool}} \xrightarrow{\text{Conv}} \xrightarrow{\text{ReLU}} \xrightarrow{\text{Pool}} \xrightarrow{\text{Flatten}} \xrightarrow{\text{Dense}} \hat{y}
$$

**Ejemplo simple: reconocer dígitos escritos a mano**

Usaremos el dataset de dígitos de sklearn (imágenes $8 \times 8$) para entender cómo una CNN procesa imágenes.

Cada imagen es una cuadrícula de $8 \times 8 = 64$ píxeles, donde cada píxel tiene un valor de intensidad entre 0 y 16:

| Pixel (0,0) | Pixel (0,1) | ... | Pixel (0,7) |
|---|---|---|---|
| 0 | 5 | ... | 0 |
| 0 | 13 | ... | 0 |
| ... | ... | ... | ... |
| 0 | 2 | ... | 0 |

El objetivo: dado un arreglo de $8 \times 8$ valores, predecir qué dígito (0-9) representa.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets

# Cargar dígitos
digits = datasets.load_digits()
X_digits = digits.images  # (1797, 8, 8)
y_digits = digits.target  # (1797,)

# Visualizar algunos ejemplos
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_digits[i], cmap='gray_r', interpolation='nearest')
    ax.set_title(f'Dígito: {y_digits[i]}', fontsize=11)
    ax.axis('off')

plt.suptitle('Ejemplos del Dataset de Dígitos (8×8 píxeles)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Forma del dataset: {X_digits.shape}")
print(f"Rango de valores: [{X_digits.min():.0f}, {X_digits.max():.0f}]")
print(f"Clases: {np.unique(y_digits)}")

## ¿Cómo procesa una CNN las imágenes?

La CNN no mira toda la imagen de golpe. En vez de eso, usa pequeños **filtros** (kernels) que se deslizan sobre la imagen detectando patrones locales. Es como mirar la imagen a través de una pequeña ventana que se va moviendo.

Pero... **¿qué es exactamente una convolución? ¿Cómo se calcula? ¿Cómo aprenden los filtros?** Para entenderlo, necesitamos la demostración matemática.

## Demostración Matemática

### Paso 1: La Operación de Convolución

En el contexto de CNNs, la "convolución" es técnicamente una **correlación cruzada** (cross-correlation). Dado:

- Una imagen de entrada $\mathbf{I}$ de tamaño $H \times W$
- Un filtro (kernel) $\mathbf{K}$ de tamaño $f \times f$

La salida (feature map) $\mathbf{O}$ se calcula como:

$$
O(i, j) = \sum_{m=0}^{f-1} \sum_{n=0}^{f-1} I(i+m, j+n) \cdot K(m, n) + b
$$

Donde:
- $(i, j)$: posición en la salida
- $(m, n)$: posición dentro del filtro
- $b$: sesgo (bias) del filtro

Visualmente, el filtro se **desliza** sobre la imagen y en cada posición calcula un **producto punto** entre el filtro y la región de la imagen que cubre:

$$
\begin{bmatrix} I_{00} & I_{01} & I_{02} \\ I_{10} & I_{11} & I_{12} \\ I_{20} & I_{21} & I_{22} \end{bmatrix} * \begin{bmatrix} K_{00} & K_{01} \\ K_{10} & K_{11} \end{bmatrix} = \begin{bmatrix} O_{00} & O_{01} \\ O_{10} & O_{11} \end{bmatrix}
$$

**Tamaño de la salida:**

$$
H_{out} = \left\lfloor \frac{H - f + 2p}{s} \right\rfloor + 1, \quad W_{out} = \left\lfloor \frac{W - f + 2p}{s} \right\rfloor + 1
$$

Donde:
- $p$: padding (ceros agregados al borde)
- $s$: stride (paso del deslizamiento)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ejemplo visual de convolución paso a paso
image = np.array([
    [1, 2, 3, 0],
    [0, 1, 2, 3],
    [3, 0, 1, 2],
    [2, 3, 0, 1]
])

kernel = np.array([
    [1, 0],
    [0, 1]
])

# Calcular convolución manualmente
H, W = image.shape
f = kernel.shape[0]
out_size = H - f + 1
output = np.zeros((out_size, out_size))

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
step = 0

for i in range(out_size):
    for j in range(out_size):
        region = image[i:i+f, j:j+f]
        output[i, j] = np.sum(region * kernel)
        
        ax = axes[step // 3][step % 3]
        
        # Dibujar imagen
        ax.imshow(image, cmap='Blues', vmin=0, vmax=3)
        for ii in range(H):
            for jj in range(W):
                ax.text(jj, ii, f'{image[ii,jj]:.0f}', ha='center', va='center', fontsize=12)
        
        # Resaltar región
        rect = mpatches.Rectangle((j-0.5, i-0.5), f, f, linewidth=3, 
                                   edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        # Cálculo
        terms = [f'{region[m,n]:.0f}×{kernel[m,n]:.0f}' for m in range(f) for n in range(f)]
        calc = ' + '.join(terms)
        ax.set_title(f'Pos ({i},{j}): {calc} = {output[i,j]:.0f}', fontsize=9, fontweight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        step += 1

# Mostrar resultado final
axes[2][2].set_visible(False)  # Ocultar el 9no subplot no usado

# Reemplazar con resultado
ax_result = fig.add_subplot(3, 3, 9)
ax_result.imshow(output, cmap='Oranges')
for ii in range(out_size):
    for jj in range(out_size):
        ax_result.text(jj, ii, f'{output[ii,jj]:.0f}', ha='center', va='center', fontsize=14, fontweight='bold')
ax_result.set_title(f'Feature Map (resultado)', fontsize=10, fontweight='bold')
ax_result.set_xticks([])
ax_result.set_yticks([])

plt.suptitle(f'Convolución paso a paso: Imagen 4×4 * Kernel 2×2 → Salida 3×3\n'
             f'Kernel = [[1, 0], [0, 1]]', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Paso 2: Filtros como Detectores de Características

Cada filtro detecta un **patrón específico**. Filtros comunes en visión por computadora incluyen:

**Detector de bordes horizontales:**
$$
K_h = \begin{bmatrix} 1 & 1 & 1 \\ 0 & 0 & 0 \\ -1 & -1 & -1 \end{bmatrix}
$$

**Detector de bordes verticales:**
$$
K_v = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}
$$

**Detección de bordes (Sobel):**
$$
K_{sobel} = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}
$$

En una CNN, **los filtros no se diseñan manualmente** — se **aprenden** automáticamente durante el entrenamiento. La red descubre qué patrones son útiles para la tarea.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demostración: aplicar filtros conocidos a un dígito
digit_img = X_digits[0]  # un dígito "0"

# Filtros predefinidos
filters = {
    'Bordes\nhorizontales': np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]]),
    'Bordes\nverticales': np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]]),
    'Sobel\nhorizontal': np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    'Esquinas': np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]),
}

def convolve2d(image, kernel):
    H, W = image.shape
    f = kernel.shape[0]
    out_h, out_w = H - f + 1, W - f + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(image[i:i+f, j:j+f] * kernel)
    return output

fig, axes = plt.subplots(1, 5, figsize=(16, 3))

axes[0].imshow(digit_img, cmap='gray_r')
axes[0].set_title(f'Original\n(dígito {y_digits[0]})', fontsize=10, fontweight='bold')
axes[0].axis('off')

for ax, (name, kernel) in zip(axes[1:], filters.items()):
    feature_map = convolve2d(digit_img, kernel)
    ax.imshow(feature_map, cmap='RdBu_r')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Efecto de diferentes filtros sobre un dígito', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Paso 3: Pooling (Submuestreo)

Después de la convolución, se aplica **pooling** para:
- **Reducir la dimensionalidad** del feature map
- Hacer la representación más **robusta a pequeños desplazamientos**
- Reducir el **costo computacional** de las capas siguientes

El tipo más común es el **Max Pooling**: en cada ventana de tamaño $p \times p$, se toma el **valor máximo**.

$$
O(i, j) = \max_{0 \leq m < p, \ 0 \leq n < p} I(i \cdot s + m, \ j \cdot s + n)
$$

**Ejemplo con Max Pooling de $2 \times 2$ y stride 2:**

$$
\begin{bmatrix} \color{red}{6} & 2 & 3 & \color{blue}{5} \\ 1 & 4 & 0 & 2 \\ \color{green}{8} & 3 & \color{orange}{7} & 1 \\ 0 & 5 & 2 & 4 \end{bmatrix} \xrightarrow{\text{Max Pool } 2 \times 2} \begin{bmatrix} \color{red}{6} & \color{blue}{5} \\ \color{green}{8} & \color{orange}{7} \end{bmatrix}
$$

La imagen se reduce de $4 \times 4$ a $2 \times 2$, conservando las **activaciones más fuertes**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ejemplo visual de Max Pooling
feature_map = np.array([
    [6, 2, 3, 5],
    [1, 4, 0, 2],
    [8, 3, 7, 1],
    [0, 5, 2, 4]
])

pool_size = 2
out_h = feature_map.shape[0] // pool_size
out_w = feature_map.shape[1] // pool_size
pooled = np.zeros((out_h, out_w))

colors = ['#e74c3c', '#3498db', '#27ae60', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Feature map original
axes[0].imshow(feature_map, cmap='YlOrRd', vmin=0, vmax=8)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, f'{feature_map[i,j]:.0f}', ha='center', va='center', fontsize=14, fontweight='bold')

# Dibujar regiones de pooling
idx = 0
for i in range(out_h):
    for j in range(out_w):
        region = feature_map[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
        pooled[i, j] = np.max(region)
        rect = mpatches.Rectangle((j*pool_size-0.5, i*pool_size-0.5), pool_size, pool_size,
                                   linewidth=3, edgecolor=colors[idx], facecolor='none')
        axes[0].add_patch(rect)
        idx += 1

axes[0].set_title('Feature Map (4×4)', fontsize=12, fontweight='bold')
axes[0].set_xticks([])
axes[0].set_yticks([])

# Resultado del pooling
axes[1].imshow(pooled, cmap='YlOrRd', vmin=0, vmax=8)
idx = 0
for i in range(out_h):
    for j in range(out_w):
        axes[1].text(j, i, f'{pooled[i,j]:.0f}', ha='center', va='center',
                    fontsize=18, fontweight='bold', color=colors[idx])
        idx += 1
axes[1].set_title('Max Pooling 2×2 (2×2)', fontsize=12, fontweight='bold')
axes[1].set_xticks([])
axes[1].set_yticks([])

plt.suptitle('Max Pooling: tomar el valor máximo de cada región', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Reducción: 4×4 = 16 valores → 2×2 = 4 valores (75% de reducción)")

### Paso 4: Forward Pass completo de una CNN

El flujo completo de datos en una CNN es:

**1. Capa Convolucional:** Aplica $n_f$ filtros de tamaño $f \times f$ a la entrada.

Para una entrada con $C$ canales (ej. 3 para RGB), cada filtro tiene dimensión $C \times f \times f$:

$$
Z^{(k)}(i,j) = \sum_{c=1}^{C} \sum_{m=0}^{f-1} \sum_{n=0}^{f-1} I^{(c)}(i+m, j+n) \cdot K^{(k,c)}(m,n) + b^{(k)}
$$

Donde $k$ indexa el filtro y $c$ el canal.

**2. Activación ReLU:**

$$
A^{(k)}(i,j) = \max(0, Z^{(k)}(i,j))
$$

**3. Max Pooling:**

$$
P^{(k)}(i,j) = \max_{\text{ventana}} A^{(k)}(\cdot, \cdot)
$$

**4. Flatten:** Convierte el tensor 3D en un vector 1D.

Si después del pooling tenemos $n_f$ feature maps de $h \times w$, el vector tiene dimensión $n_f \times h \times w$.

**5. Capa Dense (Fully Connected):**

$$
\mathbf{z} = \mathbf{W} \cdot \mathbf{a}_{\text{flat}} + \mathbf{b}
$$

**6. Softmax** (para clasificación multiclase):

$$
\hat{y}_c = \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}
$$

### Paso 5: Backpropagation en CNN

Para entrenar la CNN, necesitamos calcular los gradientes de la función de pérdida con respecto a cada parámetro. Usamos la **entropía cruzada** como pérdida:

$$
\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)
$$

#### Gradiente en la capa Dense + Softmax:

La derivada de la pérdida respecto a la pre-activación de softmax es:

$$
\frac{\partial \mathcal{L}}{\partial z_c} = \hat{y}_c - y_c
$$

Y los gradientes de los pesos de la capa dense:

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}} \cdot \mathbf{a}_{\text{flat}}^T, \quad \frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}}
$$

#### Gradiente a través de Max Pooling:

El gradiente solo pasa al elemento que fue el máximo en cada ventana:

$$
\frac{\partial \mathcal{L}}{\partial A(i',j')} = \begin{cases} \frac{\partial \mathcal{L}}{\partial P(i,j)} & \text{si } A(i',j') \text{ fue el máximo} \\ 0 & \text{en caso contrario} \end{cases}
$$

#### Gradiente en la capa Convolucional:

Para los filtros:

$$
\frac{\partial \mathcal{L}}{\partial K^{(k)}(m,n)} = \sum_{i} \sum_{j} \frac{\partial \mathcal{L}}{\partial Z^{(k)}(i,j)} \cdot I(i+m, j+n)
$$

Esto es otra convolución: la entrada convolucionada con el gradiente de la salida.

Para el bias:

$$
\frac{\partial \mathcal{L}}{\partial b^{(k)}} = \sum_{i} \sum_{j} \frac{\partial \mathcal{L}}{\partial Z^{(k)}(i,j)}
$$

### Ejemplo numérico

Veamos el forward pass completo con un ejemplo concreto.

**Entrada:** Una imagen $4 \times 4$

$$
\mathbf{I} = \begin{bmatrix} 1 & 0 & 2 & 1 \\ 0 & 3 & 1 & 0 \\ 2 & 1 & 0 & 3 \\ 1 & 0 & 2 & 1 \end{bmatrix}
$$

**Filtro** $3 \times 3$:

$$
\mathbf{K} = \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}, \quad b = 0
$$

**Paso 1 — Convolución (stride=1, no padding):**

Tamaño de salida: $(4-3+1) \times (4-3+1) = 2 \times 2$

$$
Z(0,0) = 1(1)+0(0)+2(-1)+0(1)+3(0)+1(-1)+2(1)+1(0)+0(-1) = 1+0-2+0+0-1+2+0+0 = 0
$$

$$
Z(0,1) = 0(1)+2(0)+1(-1)+3(1)+1(0)+0(-1)+1(1)+0(0)+3(-1) = 0+0-1+3+0+0+1+0-3 = 0
$$

$$
Z(1,0) = 0(1)+3(0)+1(-1)+2(1)+1(0)+0(-1)+1(1)+0(0)+2(-1) = 0+0-1+2+0+0+1+0-2 = 0
$$

$$
Z(1,1) = 3(1)+1(0)+0(-1)+1(1)+0(0)+3(-1)+0(1)+2(0)+1(-1) = 3+0+0+1+0-3+0+0-1 = 0
$$

$$
\mathbf{Z} = \begin{bmatrix} 0 & 0 \\ 0 & 0 \end{bmatrix}
$$

**Paso 2 — ReLU:** $\text{ReLU}(\mathbf{Z}) = \begin{bmatrix} 0 & 0 \\ 0 & 0 \end{bmatrix}$

**Paso 3 — Max Pool $2 \times 2$:** $\begin{bmatrix} 0 \end{bmatrix}$

(El filtro de bordes verticales no detectó bordes en esta imagen particular. Esto es esperado — en una CNN real, hay **múltiples filtros** que detectan diferentes patrones, y la red aprende los filtros que son útiles.)

In [ ]:
import numpy as np

# Ejemplo numérico completo
I = np.array([[1,0,2,1],[0,3,1,0],[2,1,0,3],[1,0,2,1]], dtype=float)
K = np.array([[1,0,-1],[1,0,-1],[1,0,-1]], dtype=float)
b = 0.0

print("FORWARD PASS COMPLETO")
print("="*50)

# Paso 1: Convolución
f = K.shape[0]
out_h = I.shape[0] - f + 1
out_w = I.shape[1] - f + 1
Z = np.zeros((out_h, out_w))

print("\n1. CONVOLUCIÓN (filtro de bordes verticales):")
for i in range(out_h):
    for j in range(out_w):
        region = I[i:i+f, j:j+f]
        Z[i,j] = np.sum(region * K) + b
        print(f"   Z({i},{j}) = Σ(región * kernel) + b = {Z[i,j]:.1f}")

print(f"\n   Z = \n{Z}")

# Paso 2: ReLU
A = np.maximum(0, Z)
print(f"\n2. ReLU(Z) = \n{A}")

# Paso 3: Max Pooling 2x2
pool_size = 2
P = np.max(A.reshape(out_h//pool_size, pool_size, out_w//pool_size, pool_size), axis=(1,3))
print(f"\n3. Max Pooling 2×2 = \n{P}")

# Paso 4: Flatten
flat = P.flatten()
print(f"\n4. Flatten = {flat}")

# Ahora con un ejemplo que SÍ detecte bordes
print("\n" + "="*50)
print("Ahora con una imagen que TIENE bordes verticales:")
print("="*50)

I2 = np.array([[0,0,5,5],[0,0,5,5],[0,0,5,5],[0,0,5,5]], dtype=float)
Z2 = np.zeros((out_h, out_w))
for i in range(out_h):
    for j in range(out_w):
        Z2[i,j] = np.sum(I2[i:i+f, j:j+f] * K) + b

A2 = np.maximum(0, Z2)
print(f"\nImagen con borde vertical:\n{I2}")
print(f"\nZ (convolución) = \n{Z2}")
print(f"\nReLU(Z) = \n{A2}")
print(f"\nMax Pool = {np.max(A2.reshape(1,2,1,2), axis=(1,3))}")
print("\n¡El filtro detectó el borde vertical con activación fuerte!")

### Número de parámetros en una CNN

Una ventaja clave de las CNNs es la **eficiencia en parámetros** comparada con redes fully-connected:

**Capa Convolucional:** con $n_f$ filtros de tamaño $f \times f$ sobre $C$ canales:

$$
\text{Parámetros} = n_f \times (C \times f \times f + 1)
$$

El $+1$ es por el bias de cada filtro.

**Ejemplo comparativo** para una imagen $28 \times 28$ (1 canal):

| Capa | Cálculo | Parámetros |
|---|---|---|
| Dense (128 neuronas) | $28 \times 28 \times 128 + 128$ | **100,480** |
| Conv (8 filtros 3×3) | $8 \times (1 \times 3 \times 3 + 1)$ | **80** |

¡La capa convolucional usa **1,256 veces menos** parámetros!

## Implementación con Código

### Funciones

In [ ]:
import numpy as np

# ==============================
# Operaciones fundamentales
# ==============================

def conv2d_forward(input, kernels, biases):
    """
    Forward pass de convolución 2D.
    input: (n_channels, H, W)
    kernels: (n_filters, n_channels, f, f)
    biases: (n_filters,)
    return: (n_filters, H_out, W_out)
    """
    n_filters, n_channels, f, _ = kernels.shape
    n_c, H, W = input.shape
    H_out = H - f + 1
    W_out = W - f + 1
    output = np.zeros((n_filters, H_out, W_out))
    
    for k in range(n_filters):
        for i in range(H_out):
            for j in range(W_out):
                region = input[:, i:i+f, j:j+f]  # (n_channels, f, f)
                output[k, i, j] = np.sum(region * kernels[k]) + biases[k]
    
    return output


def conv2d_backward(d_output, input, kernels):
    """
    Backward pass de convolución 2D.
    d_output: (n_filters, H_out, W_out) - gradiente de la salida
    input: (n_channels, H, W) - entrada guardada del forward
    kernels: (n_filters, n_channels, f, f)
    return: d_kernels, d_biases, d_input
    """
    n_filters, n_channels, f, _ = kernels.shape
    n_c, H, W = input.shape
    H_out, W_out = d_output.shape[1], d_output.shape[2]
    
    d_kernels = np.zeros_like(kernels)
    d_biases = np.zeros(n_filters)
    d_input = np.zeros_like(input)
    
    for k in range(n_filters):
        d_biases[k] = np.sum(d_output[k])
        for i in range(H_out):
            for j in range(W_out):
                region = input[:, i:i+f, j:j+f]
                d_kernels[k] += d_output[k, i, j] * region
                d_input[:, i:i+f, j:j+f] += d_output[k, i, j] * kernels[k]
    
    return d_kernels, d_biases, d_input


def maxpool_forward(input, pool_size=2):
    """
    Forward pass de Max Pooling.
    input: (n_channels, H, W)
    return: output, mask (para el backward)
    """
    n_c, H, W = input.shape
    H_out = H // pool_size
    W_out = W // pool_size
    output = np.zeros((n_c, H_out, W_out))
    mask = np.zeros_like(input)  # Guardar posiciones de los máximos
    
    for c in range(n_c):
        for i in range(H_out):
            for j in range(W_out):
                region = input[c, i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
                max_val = np.max(region)
                output[c, i, j] = max_val
                # Guardar posición del máximo
                max_mask = (region == max_val)
                mask[c, i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size] = max_mask
    
    return output, mask


def maxpool_backward(d_output, mask, pool_size=2):
    """
    Backward pass de Max Pooling.
    d_output: (n_channels, H_out, W_out)
    mask: máscara del forward
    return: d_input
    """
    n_c, H_out, W_out = d_output.shape
    d_input = np.zeros_like(mask)
    
    for c in range(n_c):
        for i in range(H_out):
            for j in range(W_out):
                d_input[c, i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size] = \
                    d_output[c, i, j] * mask[c, i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
    
    return d_input


def relu_forward(x):
    """ReLU forward"""
    return np.maximum(0, x)


def relu_backward(d_output, x):
    """ReLU backward: gradiente solo pasa donde x > 0"""
    return d_output * (x > 0)


def softmax(z):
    """Softmax estable numéricamente"""
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()


def cross_entropy_loss(y_pred, y_true):
    """Entropía cruzada"""
    return -np.log(y_pred[y_true] + 1e-10)


print("Funciones de CNN definidas correctamente.")
print("Operaciones: conv2d, maxpool, relu, softmax, cross_entropy")

In [ ]:
import numpy as np

# Verificar las funciones con un ejemplo
np.random.seed(42)

# Imagen de entrada: 1 canal, 8x8
input_img = X_digits[0].reshape(1, 8, 8) / 16.0  # Normalizar

# 4 filtros de 3x3 (1 canal de entrada)
kernels = np.random.randn(4, 1, 3, 3) * 0.5
biases = np.zeros(4)

# Forward pass
conv_out = conv2d_forward(input_img, kernels, biases)
print(f"Entrada: {input_img.shape} → (1 canal, 8×8)")
print(f"Filtros: {kernels.shape} → (4 filtros, 1 canal, 3×3)")
print(f"Después de Conv: {conv_out.shape} → (4 feature maps, 6×6)")

relu_out = relu_forward(conv_out)
print(f"Después de ReLU: {relu_out.shape}")

pool_out, pool_mask = maxpool_forward(relu_out, pool_size=2)
print(f"Después de MaxPool 2×2: {pool_out.shape} → (4 feature maps, 3×3)")

flat = pool_out.flatten()
print(f"Después de Flatten: {flat.shape} → (36 valores)")

# Capa Dense
W_dense = np.random.randn(10, flat.shape[0]) * 0.1
b_dense = np.zeros(10)
z = W_dense @ flat + b_dense
probs = softmax(z)
print(f"\nSoftmax output (probabilidades por clase):")
for c in range(10):
    bar = '█' * int(probs[c] * 50)
    print(f"  Dígito {c}: {probs[c]:.4f} {bar}")
print(f"\nPredicción: dígito {np.argmax(probs)} (real: {y_digits[0]})")

### Clase

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


class CNN:
    """
    CNN simple desde cero.
    Arquitectura: Conv → ReLU → MaxPool → Dense → Softmax
    """
    def __init__(self, n_filters=8, filter_size=3, pool_size=2, n_classes=10,
                 learning_rate=0.005):
        self.n_filters = n_filters
        self.filter_size = filter_size
        self.pool_size = pool_size
        self.n_classes = n_classes
        self.lr = learning_rate
        self.loss_history = []

        # Los pesos se inicializan en fit() cuando conocemos el tamaño de entrada
        self.kernels = None
        self.conv_biases = None
        self.W_dense = None
        self.b_dense = None

    def _init_weights(self, input_shape):
        """Inicializa pesos según el tamaño de entrada"""
        n_c, H, W = input_shape
        f = self.filter_size

        # Pesos de la capa convolucional (Xavier initialization)
        fan_in = n_c * f * f
        self.kernels = np.random.randn(self.n_filters, n_c, f, f) * np.sqrt(2.0 / fan_in)
        self.conv_biases = np.zeros(self.n_filters)

        # Calcular tamaño después de conv + pool
        conv_h = H - f + 1
        conv_w = W - f + 1
        pool_h = conv_h // self.pool_size
        pool_w = conv_w // self.pool_size
        flat_size = self.n_filters * pool_h * pool_w

        # Pesos de la capa dense
        self.W_dense = np.random.randn(self.n_classes, flat_size) * np.sqrt(2.0 / flat_size)
        self.b_dense = np.zeros(self.n_classes)

    def _softmax(self, z):
        e_z = np.exp(z - np.max(z))
        return e_z / e_z.sum()

    def _forward(self, x):
        """Forward pass completo, guarda valores intermedios para backprop"""
        cache = {}

        # Conv2D
        cache['input'] = x
        n_filters, n_c, f, _ = self.kernels.shape
        _, H, W = x.shape
        H_out = H - f + 1
        W_out = W - f + 1
        conv_out = np.zeros((n_filters, H_out, W_out))
        for k in range(n_filters):
            for i in range(H_out):
                for j in range(W_out):
                    region = x[:, i:i+f, j:j+f]
                    conv_out[k, i, j] = np.sum(region * self.kernels[k]) + self.conv_biases[k]
        cache['conv_out'] = conv_out

        # ReLU
        relu_out = np.maximum(0, conv_out)
        cache['relu_out'] = relu_out

        # Max Pooling
        p = self.pool_size
        n_c_out, H_r, W_r = relu_out.shape
        H_p = H_r // p
        W_p = W_r // p
        pool_out = np.zeros((n_c_out, H_p, W_p))
        pool_mask = np.zeros_like(relu_out)
        for c in range(n_c_out):
            for i in range(H_p):
                for j in range(W_p):
                    region = relu_out[c, i*p:(i+1)*p, j*p:(j+1)*p]
                    max_val = np.max(region)
                    pool_out[c, i, j] = max_val
                    pool_mask[c, i*p:(i+1)*p, j*p:(j+1)*p] = (region == max_val)
        cache['pool_out'] = pool_out
        cache['pool_mask'] = pool_mask

        # Flatten
        flat = pool_out.flatten()
        cache['flat'] = flat

        # Dense
        z = self.W_dense @ flat + self.b_dense
        cache['z'] = z

        # Softmax
        probs = self._softmax(z)
        cache['probs'] = probs

        return probs, cache

    def _backward(self, y_true, cache):
        """Backward pass completo"""
        probs = cache['probs']
        flat = cache['flat']
        pool_mask = cache['pool_mask']
        relu_out = cache['relu_out']
        conv_out = cache['conv_out']
        x = cache['input']

        # Gradiente de Softmax + Cross-Entropy
        d_z = probs.copy()
        d_z[y_true] -= 1  # ŷ - y

        # Gradiente de Dense
        d_W = np.outer(d_z, flat)
        d_b = d_z
        d_flat = self.W_dense.T @ d_z

        # Reshape de flat a pool shape
        d_pool = d_flat.reshape(cache['pool_out'].shape)

        # Gradiente de Max Pooling
        p = self.pool_size
        n_c_out, H_p, W_p = d_pool.shape
        d_relu = np.zeros_like(relu_out)
        for c in range(n_c_out):
            for i in range(H_p):
                for j in range(W_p):
                    d_relu[c, i*p:(i+1)*p, j*p:(j+1)*p] = \
                        d_pool[c, i, j] * pool_mask[c, i*p:(i+1)*p, j*p:(j+1)*p]

        # Gradiente de ReLU
        d_conv = d_relu * (conv_out > 0)

        # Gradiente de Conv2D
        n_filters, n_ch, f, _ = self.kernels.shape
        H_out, W_out = d_conv.shape[1], d_conv.shape[2]
        d_kernels = np.zeros_like(self.kernels)
        d_conv_biases = np.zeros(n_filters)
        for k in range(n_filters):
            d_conv_biases[k] = np.sum(d_conv[k])
            for i in range(H_out):
                for j in range(W_out):
                    region = x[:, i:i+f, j:j+f]
                    d_kernels[k] += d_conv[k, i, j] * region

        # Actualizar pesos (SGD)
        self.kernels -= self.lr * d_kernels
        self.conv_biases -= self.lr * d_conv_biases
        self.W_dense -= self.lr * d_W
        self.b_dense -= self.lr * d_b

    def fit(self, X, y, epochs=3, verbose=True):
        """
        Entrena la CNN.
        X: (n_samples, channels, H, W)
        y: (n_samples,) etiquetas enteras
        """
        if self.kernels is None:
            self._init_weights(X[0].shape)

        n_samples = len(X)
        self.loss_history = []

        for epoch in range(epochs):
            epoch_loss = 0
            correct = 0

            # Shuffle
            indices = np.random.permutation(n_samples)

            for idx in indices:
                x_i = X[idx]
                y_i = y[idx]

                # Forward
                probs, cache = self._forward(x_i)

                # Loss
                loss = -np.log(probs[y_i] + 1e-10)
                epoch_loss += loss

                if np.argmax(probs) == y_i:
                    correct += 1

                # Backward
                self._backward(y_i, cache)

            avg_loss = epoch_loss / n_samples
            accuracy = correct / n_samples
            self.loss_history.append(avg_loss)

            if verbose:
                print(f"Época {epoch+1}/{epochs} — Loss: {avg_loss:.4f} — Accuracy: {accuracy:.4f}")

    def predict(self, X):
        """Predice las clases para un batch de datos"""
        predictions = []
        for x_i in X:
            probs, _ = self._forward(x_i)
            predictions.append(np.argmax(probs))
        return np.array(predictions)

    def score(self, X, y):
        """Calcula la accuracy"""
        preds = self.predict(X)
        return np.sum(preds == y) / len(y)

    def plot_loss(self):
        """Grafica la curva de pérdida durante el entrenamiento"""
        plt.figure(figsize=(8, 4))
        plt.plot(range(1, len(self.loss_history)+1), self.loss_history,
                 'o-', color='#e74c3c', linewidth=2, markersize=6)
        plt.xlabel('Época', fontsize=12)
        plt.ylabel('Loss (Cross-Entropy)', fontsize=12)
        plt.title('Curva de Entrenamiento de la CNN', fontsize=13)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    def plot_filters(self):
        """Visualiza los filtros aprendidos"""
        n = self.n_filters
        cols = min(n, 8)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(2*cols, 2*rows))
        if rows == 1:
            axes = [axes] if n == 1 else axes
        else:
            axes = axes.flat
        for i, ax in enumerate(axes):
            if i < n:
                # Mostrar el primer canal del filtro
                ax.imshow(self.kernels[i, 0], cmap='RdBu_r', interpolation='nearest')
                ax.set_title(f'Filtro {i+1}', fontsize=9)
            ax.axis('off')
        plt.suptitle('Filtros Aprendidos por la CNN', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def plot_feature_maps(self, x, n_show=None):
        """Visualiza los feature maps para una imagen de entrada"""
        probs, cache = self._forward(x)
        conv_out = cache['conv_out']
        relu_out = cache['relu_out']

        n = n_show or self.n_filters
        fig, axes = plt.subplots(2, n + 1, figsize=(2.5*(n+1), 5))

        # Fila 1: imagen original + salida de convolución
        axes[0, 0].imshow(x[0], cmap='gray_r')
        axes[0, 0].set_title('Entrada', fontsize=9, fontweight='bold')
        axes[0, 0].axis('off')

        axes[1, 0].imshow(x[0], cmap='gray_r')
        axes[1, 0].set_title('Entrada', fontsize=9, fontweight='bold')
        axes[1, 0].axis('off')

        for i in range(n):
            axes[0, i+1].imshow(conv_out[i], cmap='RdBu_r')
            axes[0, i+1].set_title(f'Conv {i+1}', fontsize=9)
            axes[0, i+1].axis('off')

            axes[1, i+1].imshow(relu_out[i], cmap='hot')
            axes[1, i+1].set_title(f'ReLU {i+1}', fontsize=9)
            axes[1, i+1].axis('off')

        axes[0, 0].set_ylabel('Conv', fontsize=10, fontweight='bold')
        axes[1, 0].set_ylabel('ReLU', fontsize=10, fontweight='bold')
        pred = np.argmax(probs)
        plt.suptitle(f'Feature Maps — Predicción: {pred} (prob: {probs[pred]:.2f})',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
import numpy as np

# Cargar y preparar datos
digits = datasets.load_digits()
X = digits.images / 16.0  # Normalizar a [0, 1]
y = digits.target

# Agregar dimensión de canal: (n_samples, 8, 8) → (n_samples, 1, 8, 8)
X = X[:, np.newaxis, :, :]

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} imágenes")
print(f"Prueba: {X_test.shape[0]} imágenes")
print(f"Forma de cada imagen: {X_train[0].shape} (canales, alto, ancho)")

In [ ]:
# Entrenar la CNN
np.random.seed(42)
model = CNN(
    n_filters=8,
    filter_size=3,
    pool_size=2,
    n_classes=10,
    learning_rate=0.005
)

model.fit(X_train, y_train, epochs=5, verbose=True)

In [ ]:
# Evaluar
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print(f"Accuracy en entrenamiento: {train_acc:.4f}")
print(f"Accuracy en prueba: {test_acc:.4f}")

In [ ]:
# Visualizar curva de pérdida
model.plot_loss()

In [ ]:
# Visualizar filtros aprendidos
model.plot_filters()

In [ ]:
# Visualizar feature maps para algunos dígitos
for idx in [0, 10, 50]:
    model.plot_feature_maps(X_test[idx], n_show=8)

In [ ]:
# Visualizar predicciones
predictions = model.predict(X_test)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i][0], cmap='gray_r')
    pred = predictions[i]
    true = y_test[i]
    color = 'green' if pred == true else 'red'
    ax.set_title(f'P:{pred} R:{true}', fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('Predicciones de la CNN (Verde=Correcto, Rojo=Incorrecto)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de confusión
from collections import Counter

confusion = np.zeros((10, 10), dtype=int)
for true, pred in zip(y_test, predictions):
    confusion[true][pred] += 1

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(confusion, cmap='Blues')
for i in range(10):
    for j in range(10):
        color = 'white' if confusion[i, j] > confusion.max() / 2 else 'black'
        ax.text(j, i, str(confusion[i, j]), ha='center', va='center', color=color, fontsize=11)

ax.set_xlabel('Predicho', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_title('Matriz de Confusión de la CNN', fontsize=13, fontweight='bold')
plt.colorbar(im)
plt.tight_layout()
plt.show()

# Accuracy por clase
print("\nAccuracy por dígito:")
for digit in range(10):
    mask = y_test == digit
    if mask.sum() > 0:
        acc = np.sum(predictions[mask] == digit) / mask.sum()
        print(f"  Dígito {digit}: {acc:.4f} ({mask.sum()} muestras)")

In [ ]:
# Comparación: CNN vs MLP simple en los mismos datos
print("="*60)
print("COMPARACIÓN: CNN vs MLP (Fully Connected)")
print("="*60)

# MLP simple desde cero
class SimpleMLP:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0/input_size)
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2.0/hidden_size)
        self.b2 = np.zeros(output_size)
        self.lr = lr

    def _softmax(self, z):
        e_z = np.exp(z - np.max(z))
        return e_z / e_z.sum()

    def fit(self, X, y, epochs=5):
        for epoch in range(epochs):
            loss_sum = 0
            correct = 0
            indices = np.random.permutation(len(X))
            for idx in indices:
                x_i = X[idx].flatten()
                y_i = y[idx]
                # Forward
                z1 = self.W1 @ x_i + self.b1
                a1 = np.maximum(0, z1)
                z2 = self.W2 @ a1 + self.b2
                probs = self._softmax(z2)
                loss_sum += -np.log(probs[y_i] + 1e-10)
                if np.argmax(probs) == y_i:
                    correct += 1
                # Backward
                d_z2 = probs.copy()
                d_z2[y_i] -= 1
                d_W2 = np.outer(d_z2, a1)
                d_b2 = d_z2
                d_a1 = self.W2.T @ d_z2
                d_z1 = d_a1 * (z1 > 0)
                d_W1 = np.outer(d_z1, x_i)
                d_b1 = d_z1
                # Update
                self.W2 -= self.lr * d_W2
                self.b2 -= self.lr * d_b2
                self.W1 -= self.lr * d_W1
                self.b1 -= self.lr * d_b1
            print(f"  Época {epoch+1}/{epochs} — Loss: {loss_sum/len(X):.4f} — Acc: {correct/len(X):.4f}")

    def predict(self, X):
        preds = []
        for x_i in X:
            z1 = self.W1 @ x_i.flatten() + self.b1
            a1 = np.maximum(0, z1)
            z2 = self.W2 @ a1 + self.b2
            preds.append(np.argmax(z2))
        return np.array(preds)

    def score(self, X, y):
        return np.sum(self.predict(X) == y) / len(y)

# Entrenar MLP
print("\nEntrenando MLP (64 → 32 → 10):")
np.random.seed(42)
mlp = SimpleMLP(input_size=64, hidden_size=32, output_size=10, lr=0.01)
mlp.fit(X_train, y_train, epochs=5)
mlp_test_acc = mlp.score(X_test, y_test)

# Entrenar CNN
print("\nEntrenando CNN (8 filtros 3×3 → Pool → Dense):")
np.random.seed(42)
cnn = CNN(n_filters=8, filter_size=3, pool_size=2, n_classes=10, learning_rate=0.005)
cnn.fit(X_train, y_train, epochs=5)
cnn_test_acc = cnn.score(X_test, y_test)

# Contar parámetros
mlp_params = 64*32 + 32 + 32*10 + 10
cnn_conv_params = 8 * (1 * 3 * 3 + 1)
cnn_dense_params = cnn.W_dense.size + cnn.b_dense.size
cnn_total_params = cnn_conv_params + cnn_dense_params

print(f"\n{'='*60}")
print(f"{'Modelo':<20} {'Accuracy Test':<18} {'Parámetros':<15}")
print(f"{'-'*60}")
print(f"{'MLP':<20} {mlp_test_acc:<18.4f} {mlp_params:<15,}")
print(f"{'CNN':<20} {cnn_test_acc:<18.4f} {cnn_total_params:<15,}")
print(f"{'='*60}")